In [2]:
import cv2
import mediapipe as mp
import warnings
import numpy as np
import math
import pyautogui
import time

warnings.filterwarnings("ignore")

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
cap = cv2.VideoCapture(0)

screen_width, screen_height = pyautogui.size()
frame = 150
prev_x, prev_y = 0,0
now_x, now_y = 0,0
smooth = 3 

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break
            
        image = cv2.flip(image, 1)
        h, w, _ = image.shape
        
        image.flags.writeable = False
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_image)
        image.flags.writeable = True

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS
                )

                landmarks = hand_landmarks.landmark

                # 주요 포인트
                thumb_tip = landmarks[4]
                index_tip = landmarks[8]
                index_pip = landmarks[6]
                middle_tip = landmarks[12]
                middle_pip = landmarks[10]
                ring_tip = landmarks[16]
                ring_pip = landmarks[14]
                pinky_tip = landmarks[20]
                pinky_pip = landmarks[18]
                wrist = landmarks[0]

                ix, iy = int(index_tip.x * w), int(index_tip.y * h)
                tx, ty = int(thumb_tip.x * w), int(thumb_tip.y * h)
                mx, my = int(middle_tip.x * w), int(middle_tip.y * h)
                px, py = int(pinky_tip.x * w), int(pinky_tip.y * h)
                
                # 부드럽게 만들기
                mouse_x = np.interp(ix, [frame, w-frame], [0, screen_width])
                mouse_y = np.interp(iy, [frame, h-frame], [0, screen_height])
                
                now_x = prev_x + (mouse_x - prev_x) / smooth
                now_y = prev_y + (mouse_y - prev_y) / smooth

                pyautogui.moveTo(now_x, now_y, _pause=False)

                
                # 1. 좌클릭 (엄지-검지)
                index_dis = np.sqrt((ix - tx)**2 + (iy - ty)**2)
                
                # 2. 우클릭 (엄지-중지)
                middle_dis = np.sqrt((mx - tx)**2 + (my - ty)**2)

                # 3. 드래그 (엄지-새끼)
                pinky_dis = np.sqrt((px - tx)**2 + (py - ty)**2)

                click = index_dis < 35
                double_click = middle_dis < 35
                hold_time = 0.3
                drag_start_time = None
                dragging = False
                v_shape = index_tip.y < index_pip.y and middle_tip.y < middle_pip.y 
                            and ring_tip.y > ring_pip.y and pinky_tip.y > pinky_pip.y
            
    
                if click:
                    pyautogui.click()
                    cv2.putText(image, "CLICK!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    time.sleep(0.3)

                elif double_click:
                    pyautogui.doubleClick()
                    cv2.putText(image, "DOUBLE CLICK", (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    time.sleep(0.3)

                if pinky_dis < 35:
                    if drag_start_time is None:
                        drag_start_time = time.time()

                    elif time.time() - drag_start_time > hold_time:
                        if not dragging:
                            pyautogui.mouseDown()
                            cv2.putText(image, "DRAG", (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                            dragging = True

                else:
                    drag_start_time = None
                    if dragging:
                        pyautogui.mouseUp()
                        dragging = False

                if 
                            
                            
                    
                # 픽셀 좌표
                # thumb_pt = (int(thumb_tip.x * w), int(thumb_tip.y * h))
                # index_pt = (int(index_tip.x * w), int(index_tip.y * h))
                # index_pip_pt = (int(index_pip.x * w), int(index_pip.y * h))
                # wrist_pt = (int(wrist.x * w), int(wrist.y * h))
                # middle_pt = (int(middle_tip.x * w), int(middle_tip.y * h))
                # middle_pip_pt = (int(middle_pip.x * w), int(middle_pip.y * h))
      

        cv2.imshow('Hands', image)
        key = cv2.waitKey(10) & 0xFF 
        if key == ord('q') or key == ord('Q'):
            break

cap.release()
cv2.destroyAllWindows()

In [4]:
import cv2
import mediapipe as mp
import warnings
import numpy as np
import pyautogui
import time
import math

warnings.filterwarnings("ignore")

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("카메라를 열 수 없습니다.")
    exit()

# 카메라 해상도
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# 화면 해상도
screen_width, screen_height = pyautogui.size()

# pyautogui 구석 failsafe 해제
pyautogui.FAILSAFE = False

# 커서 이동 smoothing
prev_x, prev_y = 0, 0
alpha = 0.25

# 제어 프레임 여백
frame_margin = 80

# pinch / drag / double click 설정
pinch_threshold = 35          # 엄지-검지 거리 기준
middle_threshold = 35         # 엄지-중지 거리 기준
pinky_threshold = 35
drag_hold_time = 0.5          # 이 시간 이상 pinch 유지 시 드래그

# 상태 변수
pinching = False
dragging = False
click_ready = False
pinch_start_time = None

last_double_click_time = 0
double_click_delay = 0.6

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while True:
        success, image = cap.read()
        if not success:
            print("프레임을 읽을 수 없습니다.")
            break

        image = cv2.flip(image, 1)
        h, w, _ = image.shape

        usable_x_min = frame_margin
        usable_x_max = w - frame_margin
        usable_y_min = frame_margin
        usable_y_max = h - frame_margin

        # 제어 프레임 표시
        cv2.rectangle(
            image,
            (usable_x_min, usable_y_min),
            (usable_x_max, usable_y_max),
            (255, 0, 0),
            2
        )

        cv2.putText(
            image,
            "Mouse Control Area",
            (usable_x_min, usable_y_min - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_image)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS
                )

                landmarks = hand_landmarks.landmark

                thumb_tip = landmarks[4]
                index_tip = landmarks[8]
                middle_tip = landmarks[12]
                pinky_tip = landmarks[20]

                tx, ty = int(thumb_tip.x * w), int(thumb_tip.y * h)
                ix, iy = int(index_tip.x * w), int(index_tip.y * h)
                mx, my = int(middle_tip.x * w), int(middle_tip.y * h)
                px, py = int(pinky_tip.x * w), int(pinky_tip.y * h)

                # 시각화
                cv2.circle(image, (tx, ty), 5, (255, 0, 255), cv2.FILLED)   # 엄지
                cv2.circle(image, (ix, iy), 5, (0, 255, 255), cv2.FILLED)   # 검지
                cv2.circle(image, (mx, my), 5, (255, 255, 0), cv2.FILLED)   # 중지
                cv2.circle(image, (px, py), 5, (100, 255, 100), cv2.FILLED)   # 새끼
                


                # -----------------------------------
                # 검지로 마우스 이동
                # -----------------------------------
                inside_area = usable_x_min <= ix <= usable_x_max and usable_y_min <= iy <= usable_y_max

                if inside_area:
                    screen_x = np.interp(ix, [usable_x_min, usable_x_max], [0, screen_width])
                    screen_y = np.interp(iy, [usable_y_min, usable_y_max], [0, screen_height])

                    smooth_x = prev_x + (screen_x - prev_x) * alpha
                    smooth_y = prev_y + (screen_y - prev_y) * alpha

                    pyautogui.moveTo(smooth_x, smooth_y)

                    prev_x, prev_y = smooth_x, smooth_y


                # -----------------------------------
                # 거리 계산
                # -----------------------------------
                dist_index = math.hypot(ix - tx, iy - ty)   # 엄지-검지
                dist_middle = math.hypot(mx - tx, my - ty)  # 엄지-중지
                dist_pinky = math.hypot(px - tx, py - ty)


                current_time = time.time()

                # -----------------------------------
                # 1) 엄지 + 검지 : 클릭 ONLY
                # -----------------------------------
                if dist_index < pinch_threshold:
                    if not pinching:
                        pinching = True
                        pinch_start_time = current_time
                        click_ready = True
                
                else:
                    if pinching:
                        hold_time = current_time - pinch_start_time
                
                        if click_ready:
                            pyautogui.click()
                            cv2.putText(
                                image, "CLICK!",
                                (20, 190),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2
                            )
                
                        pinching = False
                        pinch_start_time = None
                        click_ready = False
                
                
                # -----------------------------------
                # 2) 엄지 + 새끼 : 드래그
                # -----------------------------------
                if dist_pinky < pinky_threshold:
                
                    if not dragging:
                        pyautogui.mouseDown()
                        dragging = True
                
                    cv2.putText(
                        image, "DRAGGING",
                        (20, 150),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2
                    )
                
                else:
                    if dragging:
                        pyautogui.mouseUp()
                        dragging = False

                # -----------------------------------
                # 3) 엄지 + 중지 : 더블클릭
                #    단, pinch/drag 중에는 막기
                # -----------------------------------
                if (dist_middle < middle_threshold) and (not pinching) and (not dragging):
                    if current_time - last_double_click_time > double_click_delay:
                        pyautogui.doubleClick()
                        last_double_click_time = current_time

                        cv2.putText(
                            image, "DOUBLE CLICK",
                            (20, 230),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 255), 2
                        )

                # 손 하나만 처리
                break

        else:
            # 손이 사라졌는데 드래그 중이면 안전하게 해제
            if dragging:
                pyautogui.mouseUp()
                dragging = False

            pinching = False
            pinch_start_time = None
            click_ready = False

        cv2.imshow("Hands Mouse Control", image)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [5]:
import cv2
import mediapipe as mp
import warnings
import numpy as np
import pyautogui
import time
import math

warnings.filterwarnings("ignore")

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("카메라를 열 수 없습니다.")
    exit()

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

screen_width, screen_height = pyautogui.size()

pyautogui.FAILSAFE = False

prev_x, prev_y = 0, 0
alpha = 0.25

frame_margin = 80

pinch_threshold = 35
middle_threshold = 35
pinky_threshold = 35

pinching = False
dragging = False
click_ready = False
pinch_start_time = None

last_double_click_time = 0
double_click_delay = 0.6

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while True:
        success, image = cap.read()
        if not success:
            print("프레임을 읽을 수 없습니다.")
            break

        image = cv2.flip(image, 1)
        h, w, _ = image.shape

        usable_x_min = frame_margin
        usable_x_max = w - frame_margin
        usable_y_min = frame_margin
        usable_y_max = h - frame_margin

        # ----------------------------
        # 제어 프레임 표시
        # ----------------------------
        cv2.rectangle(image,
                      (usable_x_min, usable_y_min),
                      (usable_x_max, usable_y_max),
                      (255, 0, 0), 2)

        cv2.putText(image,
                    "Mouse Control Area",
                    (usable_x_min, usable_y_min - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 0, 0), 2)

        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_image)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS
                )

                landmarks = hand_landmarks.landmark

                thumb_tip = landmarks[4]
                index_tip = landmarks[8]
                middle_tip = landmarks[12]
                pinky_tip = landmarks[20]

                tx, ty = int(thumb_tip.x * w), int(thumb_tip.y * h)
                ix, iy = int(index_tip.x * w), int(index_tip.y * h)
                mx, my = int(middle_tip.x * w), int(middle_tip.y * h)
                px, py = int(pinky_tip.x * w), int(pinky_tip.y * h)

                # ----------------------------
                # 시각화
                # ----------------------------
                cv2.circle(image, (tx, ty), 5, (255, 0, 255), cv2.FILLED)
                cv2.circle(image, (ix, iy), 5, (0, 255, 255), cv2.FILLED)
                cv2.circle(image, (mx, my), 5, (255, 255, 0), cv2.FILLED)
                cv2.circle(image, (px, py), 5, (100, 255, 100), cv2.FILLED)

                # ----------------------------
                # 마우스 이동
                # ----------------------------
                inside_area = (usable_x_min <= ix <= usable_x_max and
                               usable_y_min <= iy <= usable_y_max)

                if inside_area:
                    screen_x = np.interp(ix, [usable_x_min, usable_x_max], [0, screen_width])
                    screen_y = np.interp(iy, [usable_y_min, usable_y_max], [0, screen_height])

                    smooth_x = prev_x + (screen_x - prev_x) * alpha
                    smooth_y = prev_y + (screen_y - prev_y) * alpha

                    pyautogui.moveTo(smooth_x, smooth_y)

                    prev_x, prev_y = smooth_x, smooth_y

                # ----------------------------
                # 거리 계산
                # ----------------------------
                dist_index = math.hypot(ix - tx, iy - ty)
                dist_middle = math.hypot(mx - tx, my - ty)
                dist_pinky = math.hypot(px - tx, py - ty)

                current_time = time.time()

                # ----------------------------
                # 1) 엄지 + 검지 : 클릭
                # ----------------------------
                if dist_index < pinch_threshold:
                    if not pinching:
                        pinching = True
                        pinch_start_time = current_time
                        click_ready = True

                else:
                    if pinching:
                        if click_ready:
                            pyautogui.click()
                            cv2.putText(image, "CLICK!",
                                        (20, 190),
                                        cv2.FONT_HERSHEY_SIMPLEX,
                                        1, (0, 255, 0), 2)

                        pinching = False
                        pinch_start_time = None
                        click_ready = False

                # ----------------------------
                # 2) 엄지 + 새끼 : 드래그
                # ----------------------------
                if dist_pinky < pinky_threshold:

                    if not dragging:
                        pyautogui.mouseDown()
                        dragging = True

                    cv2.putText(image, "DRAGGING",
                                (20, 150),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                1, (0, 0, 255), 2)

                else:
                    if dragging:
                        pyautogui.mouseUp()
                        dragging = False

                # ----------------------------
                # 3) 엄지 + 중지 : 더블클릭
                # ----------------------------
                if (dist_middle < middle_threshold) and (not pinching) and (not dragging):
                    if current_time - last_double_click_time > double_click_delay:
                        pyautogui.doubleClick()
                        last_double_click_time = current_time

                        cv2.putText(image, "DOUBLE CLICK",
                                    (20, 230),
                                    cv2.FONT_HERSHEY_SIMPLEX,
                                    1, (255, 0, 255), 2)

                break

        else:
            if dragging:
                pyautogui.mouseUp()
                dragging = False

            pinching = False
            pinch_start_time = None
            click_ready = False

        cv2.imshow("Hands Mouse Control", image)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()